# 10 — Deep Learning untuk Inspeksi Citra Tablet

### Dari geometri dan fitur rekayasa menuju jaringan konvolusional yang dapat diaudit

**Seri:** Pengolahan Citra Digital dan *Scientific Imaging* · **Bahasa:** Indonesia · **Prasyarat:** Notebook 01–09, aljabar linear, turunan parsial, peluang, serta Python dasar.

Pada notebook 09 kita membandingkan piksel, HOG dan beberapa *classifier* pada citra angka tulisan tangan. Pada notebook ini kita menghadapi persoalan yang lebih dekat dengan instrumentasi: **memutuskan apakah citra tablet memuat cacat geometris atau tekstural**, ketika iluminasi, blur, posisi, dan batch akuisisi berubah. Apakah arsitektur CNN selalu lebih baik daripada model dengan fitur yang dirancang berdasarkan pengetahuan fisik? Kapan performa yang tampak tinggi ternyata mengandalkan petunjuk visual yang rapuh?

Notebook ini melanjutkan studi kasus tablet dari repository asli. Kita mempertahankan model klasik sebagai pembanding, lalu membangun CNN, menguji *ablation* dan regularisasi, menganalisis biaya komputasi, dan mengukur robustnes dengan batas klaim yang jelas. **Seluruh data yang digunakan adalah citra tablet sintetis**, bukan citra produk farmasi nyata, dan tidak ada hasil yang boleh dianggap sebagai validasi produksi.

**Alur pembelajaran:** fenomena dan pembentukan citra → fitur berbasis pengukuran → konvolusi dan fungsi kerugian → pelatihan dan pemilihan model → evaluasi *held-out* → *ablation*, robustness, ketidakpastian → rencana validasi pada citra nyata.

## Peta capaian dan pertanyaan diagnostik

Setelah mempelajari notebook ini, pembaca mampu:

1. Memformulasikan klasifikasi inspeksi sebagai masalah keputusan dengan dua kelas dan konsekuensi kesalahan yang berbeda.
2. Menjelaskan hubungan model fisika pencitraan, model noise, dan kemungkinan *shortcut learning* pada model prediktif.
3. Menurunkan operasi konvolusi, ukuran keluaran, jumlah parameter, *cross-entropy*, dan gradien *backpropagation* sederhana.
4. Membandingkan baseline berbasis fitur geometri/tekstur dengan CNN pada data dan metrik evaluasi yang konsisten.
5. Merancang pemisahan **train–validation–test berdasarkan batch**, bukan secara acak per citra.
6. Menguji pengaruh augmentasi, kapasitas model, dan dropout melalui *ablation* yang terkontrol.
7. Melakukan evaluasi tingkat klasifikasi dan biaya kesalahan, kalibrasi probabilitas, serta *domain stress test*.
8. Mengukur ketidakpastian melalui bootstrap tingkat batch dan menyatakan batas generalisasi secara jujur.

**Diagnosis awal:** Jika tiap batch memiliki pencahayaan tertentu, mengapa *random split* per citra dapat memberi perkiraan performa yang terlalu optimistis? Mengapa CNN memerlukan *inductive bias* spasial? Apakah akurasi tinggi sudah cukup ketika *false negative* lebih mahal?

## 1. Kontrak eksperimen dan lingkungan

Notebook dapat dijalankan mandiri melalui Google Colab atau Jupyter tanpa mengunduh dataset. Pustaka yang digunakan ialah NumPy, SciPy, Matplotlib, pandas, scikit-image, scikit-learn, dan PyTorch; GPU **tidak wajib**. Pada Colab, pilih *Runtime → Run all*; di notebook lokal pastikan paket-paket tersebut tersedia.

Seluruh pembagian dilakukan **di tingkat batch** dan dikunci sebelum pelatihan. Data validasi dipakai untuk pemilihan arsitektur/hiperparameter. Data test hanya dipakai setelah pemilihan selesai dan tidak boleh digunakan untuk mengubah konfigurasi berikutnya. *Stress test* menggunakan turunan citra test, sehingga tidak merupakan sampel independen baru.

Pengendalian *random seed* membantu pengulangan pada lingkungan tertentu, tetapi tidak menjamin hasil identik lintas versi perangkat lunak atau antara CPU dan GPU. Catat seluruh versi yang tercetak.

In [ ]:
import time, random, platform, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import measure, morphology, feature, filters
from skimage.feature import hog
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                             recall_score, precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, brier_score_loss)
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 20261112
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(2, torch.get_num_threads()))
if torch.cuda.is_available():
    DEVICE=torch.device('cuda')
else:
    DEVICE=torch.device('cpu')
if hasattr(torch.backends,'cudnn'):
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
plt.rcParams.update({'figure.figsize':(9,4.3),'figure.dpi':105})
print('Python:',platform.python_version(),'| NumPy:',np.__version__,
      '| PyTorch:',torch.__version__,'| device:',DEVICE)

## 2. Masalah ilmiah: klasifikasi tidak identik dengan pengukuran

Misalkan citra tablet yang diterima sistem inspeksi adalah $Y\in[0,1]^{H\times W}$ dan label benar adalah $t\in\{0,1\}$, dengan $t=1$ menyatakan cacat. Tujuan *classifier* adalah menghasilkan peluang prediksi $p_\theta(t=1\mid Y)$ dan suatu keputusan sesuai aturan biaya. Namun keberadaan label **tidak** menghilangkan pengaruh sensor, pencahayaan, atau aturan anotasi.

Salah satu model generatif sederhana adalah

$$
Y=\mathcal Q\Big\{a_b\,[h_{\sigma_b}*R(M,D,L_b)] + d_b + \eta\Big\},
$$

dengan $R$ citra reflektansi sintetis bergantung pada bentuk tablet $M$, pola cacat $D$ dan iluminasi $L_b$; $h_{\sigma_b}$ ialah PSF blur; $a_b$ dan $d_b$ adalah gain dan offset batch; $\eta$ noise; dan $\mathcal Q$ menunjukkan clipping serta kuantisasi.

Model ini tidak identik dengan kamera nyata: respons spektral, *demosaicing*, geometri tiga dimensi, dan variasi material tidak dimasukkan secara lengkap. Kita sengaja mengendalikan variasi karena ingin mengisolasi sifat metode, bukan meniru validasi farmasi.

**Pertanyaan penting:** model yang tampak mempelajari cacat bisa saja mempelajari karakter iluminasi jika distribusi kelas berbeda antarbatches. Oleh sebab itu kelas dalam setiap batch akan diseimbangkan dan pemisahan dilakukan menurut batch.

## 3. Membangun dataset dengan asal-usul yang jelas

Kita buat tablet tampak atas sebagai cakram hampir eliptis di atas latar yang tidak sepenuhnya seragam. Cacat dibentuk sebagai salah satu dari: *chip* pada tepi, *pitting* pada permukaan, atau retakan tipis. Parameter ukur fisik di sini **hanya skala simulasi**; tidak ada konversi piksel–mm yang diperoleh dari kalibrasi.

Unit independen desain simulasi adalah spesimen yang unik, dikelompokkan ke dalam batch yang memiliki kondisi akuisisi bersama. Satu citra per spesimen dipakai untuk menghindari duplikasi frame lintas split.

In [ ]:
def sintetis_tablet(n_batch=24, per_batch=12, ukuran=40, seed=SEED):
    """Kembalikan citra float32 [0,1], label cacat, batch, dan tipe cacat.
    Dataset sepenuhnya sintetis; tiap batch berisi kelas seimbang.
    """
    rng=np.random.default_rng(seed)
    yy,xx=np.mgrid[:ukuran,:ukuran]
    X=[]; Y=[]; B=[]; T=[]
    for b in range(n_batch):
        gain=rng.uniform(0.89,1.11)
        offset=rng.uniform(-0.045,0.045)
        gx=rng.uniform(-0.12,0.12)
        gy=rng.uniform(-0.08,0.08)
        blur=rng.uniform(0.5,1.25)
        noise_sigma=rng.uniform(0.013,0.035)
        order=rng.permutation(np.repeat([0,1],per_batch//2))
        for k,label in enumerate(order):
            cx=ukuran/2+rng.normal(0,1.55)
            cy=ukuran/2+rng.normal(0,1.55)
            rad=rng.uniform(11.3,13.3)
            ar=rng.uniform(0.94,1.06)
            rr=np.sqrt(((xx-cx)/(rad*ar))**2+((yy-cy)/(rad/ar))**2)
            mask=rr<=1
            surface=0.56+0.055*(1-rr**2)+rng.normal(0,0.011,(ukuran,ukuran))
            img=np.full((ukuran,ukuran),0.16,dtype=float)
            img[mask]=surface[mask]
            tipe='utuh'
            if label:
                tipe=['chip','pitting','retak'][rng.integers(0,3)]
                if tipe=='chip':
                    angle=rng.uniform(0,2*np.pi)
                    x0=cx+rad*np.cos(angle); y0=cy+rad*np.sin(angle)
                    defect=((xx-x0)**2+(yy-y0)**2)<rng.uniform(2.0,3.5)**2
                    img[defect]=0.16
                elif tipe=='pitting':
                    phi=rng.uniform(0,2*np.pi); rho=rng.uniform(0.1,0.58)*rad
                    x0=cx+rho*np.cos(phi); y0=cy+rho*np.sin(phi)
                    defect=((xx-x0)**2+(yy-y0)**2)<rng.uniform(1.5,2.8)**2
                    img[defect & mask]*=rng.uniform(0.38,0.60)
                else:
                    angle=rng.uniform(0,2*np.pi)
                    xc=cx+rng.uniform(-3,3); yc=cy+rng.uniform(-3,3)
                    along=(xx-xc)*np.cos(angle)+(yy-yc)*np.sin(angle)
                    normal=-(xx-xc)*np.sin(angle)+(yy-yc)*np.cos(angle)
                    crack=(np.abs(normal)<rng.uniform(0.55,0.85)) & (np.abs(along)<rng.uniform(5,8)) & mask
                    img[crack]*=rng.uniform(0.35,0.55)
            illum=1+gx*(xx-ukuran/2)/ukuran+gy*(yy-ukuran/2)/ukuran
            img=ndi.gaussian_filter(img*illum,blur,mode='reflect')*gain+offset
            img += rng.normal(0,noise_sigma,(ukuran,ukuran))
            img=np.clip(np.round(np.clip(img,0,1)*255)/255,0,1)
            X.append(img.astype(np.float32)); Y.append(int(label)); B.append(b); T.append(tipe)
    return np.stack(X),np.array(Y),np.array(B),np.array(T)

images, labels, groups, defect_types=sintetis_tablet()
assert images.shape==(288,40,40)
assert np.isfinite(images).all() and set(np.unique(labels))=={0,1}
print('Jumlah citra:',len(images),'| batch:',len(np.unique(groups)),
      '| kelas:',dict(zip(*np.unique(labels,return_counts=True))))

In [ ]:
fig,ax=plt.subplots(3,6,figsize=(13,6.3))
for a,ix in zip(ax.flat, np.r_[np.where(labels==0)[0][:9],np.where(labels==1)[0][:9]]):
    a.imshow(images[ix],cmap='gray',vmin=0,vmax=1)
    a.set_title(f'{defect_types[ix]} | B{groups[ix]:02d}',fontsize=8)
    a.axis('off')
fig.suptitle('Citra tablet sintetis: variasi bentuk, cacat, dan batch')
plt.tight_layout();plt.show()

**Interpretasi:** citra terlihat mirip antarspesimen tetapi sinyal cacat tidak selalu besar dibanding noise dan blur. Karena pembangkit data diketahui, kita dapat menguji *stress shift* secara eksplisit. Namun dataset tidak memuat heterogenitas material nyata, manusia anotator, atau pengaruh kebijakan penerimaan industri.

### 3.1 Unit pemisahan dan penyebab leakage

Jika sebuah batch $b$ menghasilkan beberapa citra $Y_{bi}$ yang berbagi parameter $a_b$, $d_b$, dan $\sigma_b$, maka sampel dalam batch tidak independen secara penuh. Pembagian acak pada tingkat citra akan menyebarkan karakter akuisisi yang sama ke train dan test. Kita gunakan pemisahan kelompok: **batch test dan validation tidak pernah muncul dalam train**.

In [ ]:
gss1=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=SEED)
trainval_idx,test_idx=next(gss1.split(images,labels,groups))
gss2=GroupShuffleSplit(n_splits=1,test_size=1/3,random_state=SEED+1)
loc_tr,loc_va=next(gss2.split(images[trainval_idx],labels[trainval_idx],groups[trainval_idx]))
train_idx=trainval_idx[loc_tr]; val_idx=trainval_idx[loc_va]
assert len(set(groups[train_idx]) & set(groups[val_idx]))==0
assert len(set(groups[train_idx]) & set(groups[test_idx]))==0
assert len(set(groups[val_idx]) & set(groups[test_idx]))==0
for s,idx in [('train',train_idx),('validation',val_idx),('test',test_idx)]:
    print(f'{s:<11}: {len(idx):3d} citra | {len(np.unique(groups[idx]))} batch | cacat {labels[idx].sum()}')

### 3.2 Memastikan data sintetis tidak menyelundupkan label melalui metadata

Generator memang mengetahui jenis cacat, gain, dan kondisi batch. Informasi tersebut hanya digunakan untuk **membangun** citra dan audit distribusi. Model menerima satu objek citra atau fitur yang dihitung dari objek tersebut. Tidak ada parameter `batch`, `jenis_cacat`, maupun masker ground truth yang dipakai sebagai prediktor; memasukkannya akan mengubah masalah menjadi pengenalan aturan generator.

In [ ]:
fig,ax=plt.subplots(figsize=(9.3,2.9))
counts=np.array([[np.sum((groups==b)&(labels==0)),np.sum((groups==b)&(labels==1))]
                 for b in np.unique(groups)])
assert np.all(counts[:,0]==counts[:,1])
ax.bar(np.unique(groups),counts[:,0],label='Utuh')
ax.bar(np.unique(groups),counts[:,1],bottom=counts[:,0],label='Cacat')
ax.set(xlabel='ID batch sintetis',ylabel='Jumlah citra',title='Keseimbangan kelas pada setiap batch')
ax.legend();plt.tight_layout();plt.show()

## 4. Baseline: fitur yang berlandaskan geometri dan tekstur

Sebelum melatih CNN, baseline yang jujur diperlukan. Fitur yang mudah ditafsirkan meliputi luas objek terang, sirkularitas, perbedaan intensitas tablet–latar, variasi tekstur, dan energi gradien. Fitur diperoleh **hanya dari citra yang diamati**; label pembangkit, mask laten, identitas batch, dan jenis cacat *tidak* digunakan sebagai masukan model.

Untuk area $A$ dan keliling $P$, salah satu indikator bentuk ialah

$$
C=\frac{4\pi A}{P^2},\qquad 0<C\lesssim1,
$$

yang sensitif pada diskretisasi tepi dan resolusi. Nilai $C$ bisa lebih dari 1 akibat bias estimator keliling untuk obyek digital kecil, sehingga rumus ini bukan batas matematis ketat pada implementasi piksel.

In [ ]:
def fitur_geometri(img):
    # Pemrosesan lokal terdefinisi a priori; tidak belajar ambang dari test.
    p=filters.threshold_otsu(img)
    raw=(img>p); lab=measure.label(raw)
    sizes=np.bincount(lab.ravel())
    if len(sizes)<=1: return np.zeros(8,dtype=np.float64)
    sizes[0]=0
    mask=lab==int(np.argmax(sizes))
    regs=measure.regionprops(measure.label(mask))
    if not regs:
        return np.zeros(8,dtype=np.float64)
    reg=max(regs,key=lambda r:r.area)
    area=float(reg.area)
    circ=4*np.pi*area/(max(float(reg.perimeter),1.0)**2)
    gray=img[mask]
    sob=filters.sobel(img)
    return np.array([area,circ,reg.eccentricity,reg.solidity,
                     gray.mean(),gray.std(),np.quantile(gray,0.12),
                     sob[mask].mean()],dtype=np.float64)

Xtab=np.stack([fitur_geometri(im) for im in images])
print('Matriks fitur:',Xtab.shape,'| contoh:',np.round(Xtab[0],3))
fig,ax=plt.subplots(1,2,figsize=(10,3.6))
for cls,name in [(0,'Utuh'),(1,'Cacat')]:
    ax[0].scatter(Xtab[labels==cls,0],Xtab[labels==cls,1],s=15,alpha=.5,label=name)
    ax[1].hist(Xtab[labels==cls,5],bins=22,alpha=.45,label=name)
ax[0].set(xlabel='Luas objek (piksel²)',ylabel='Sirkularitas')
ax[1].set(xlabel='Simpangan baku sinyal objek',ylabel='Frekuensi')
for a in ax:a.legend()
plt.tight_layout();plt.show()

### 4.1 Standardisasi dan estimasi parameter

Model regresi logistik bekerja pada $\mathbf z=(\mathbf x-\boldsymbol\mu_{\rm train})/\boldsymbol\sigma_{\rm train}$. Parameter statistik **harus di-*fit* pada train saja**, termasuk ketika fitur berasal dari kode deterministik. Kita gunakan `Pipeline(StandardScaler(), LogisticRegression(...))`, yang menerapkan kontrak ini secara otomatis. SVM dengan fitur HOG dipakai sebagai baseline alternatif dalam latihan, sedangkan perbandingan utama notebook menggunakan model geometris versus CNN agar biaya komputasi tetap terjangkau.

In [ ]:
classical_candidates=[]
for Cval in [0.1,1.0,10.0]:
    clf=make_pipeline(StandardScaler(),LogisticRegression(C=Cval,max_iter=1500,random_state=SEED))
    clf.fit(Xtab[train_idx],labels[train_idx])
    pred=clf.predict(Xtab[val_idx]);score=f1_score(labels[val_idx],pred,average='macro',zero_division=0)
    classical_candidates.append((score,Cval,clf))
score_classic,Cclassic,model_classic=max(classical_candidates,key=lambda x:x[0])
print('Baseline klasik: C=',Cclassic,'| F1 val =',round(score_classic,3))

## 5. Dari konvolusi ke representasi yang dipelajari

Untuk masukan $X$ dan kernel $K$ berukuran $k_h\times k_w$, operasi yang lazim diimplementasikan oleh `nn.Conv2d` adalah **korelasi silang** (tanpa membalik kernel):

$$
Z_{o,i,j}=b_o+\sum_{c=1}^{C_{\rm in}}\sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1}K_{o,c,u,v}\,X_{c,i+u,j+v}.
$$

Dalam penggunaan deep learning operasi ini sering disebut *convolution* karena pembalikan kernel tidak membatasi keluarga filter yang bisa dipelajari. Untuk stride $s$, padding $p$, dan dilasi $d$, ukuran keluaran satu dimensi ialah

$$
H_{\rm out}=\left\lfloor\frac{H+2p-d(k_h-1)-1}{s}+1\right\rfloor.
$$

Jumlah bobot dan bias pada `Conv2d` dengan $C_{\rm in}$ kanal masukan serta $C_{\rm out}$ kanal keluaran ialah

$$
N_{\rm param}=C_{\rm out}(C_{\rm in}k_hk_w+1).
$$

Keuntungan utama di sini adalah *weight sharing*: satu kernel memindai lokasi berbeda, sehingga jumlah parameternya tidak naik secara linear dengan ukuran citra seperti pada dense layer. Hal ini memberi *inductive bias* translasi lokal, **bukan invariansi sempurna terhadap translasi**.

In [ ]:
# Menguji klaim bentuk tensor dan jumlah parameter konvolusi.
conv_demo=nn.Conv2d(1,8,kernel_size=3,padding=1)
toy=torch.randn(2,1,40,40)
out=conv_demo(toy)
print('Bentuk:',tuple(toy.shape),'→',tuple(out.shape))
print('Parameter:',sum(p.numel() for p in conv_demo.parameters()),'= 8 × (1 × 3 × 3 + 1)')
assert out.shape==(2,8,40,40)
assert sum(p.numel() for p in conv_demo.parameters())==80

### 5.1 Aktivasi, pooling, dan *receptive field*

Fungsi aktivasi `ReLU(z)=\max(0,z)` memungkinkan komposisi nonlinier. *Pooling* merangkum representasi sehingga sejumlah perubahan lokal menjadi kurang berpengaruh, tetapi juga menghilangkan informasi posisi halus. Pada inspeksi cacat kecil, downsampling terlalu dini dapat menghapus sinyal yang seharusnya diukur.

Untuk beberapa layer dengan kernel $k_l$ dan langkah efektif sebelumnya $j_{l-1}$, *receptive field* dapat dihitung secara rekursif:

$$
r_l=r_{l-1}+(k_l-1)j_{l-1},\qquad j_l=j_{l-1}s_l,
$$

dengan $r_0=j_0=1$. Rumus ini adalah *theoretical receptive field*; sensitivitas efektif terhadap piksel tidak otomatis seragam.

## 6. Fungsi objektif dan optimisasi

Untuk label biner $t\in\{0,1\}$ dan logit $z_\theta(Y)$, probabilitas model didefinisikan sebagai $p_\theta=\sigma(z_\theta)$, $\sigma(z)=1/(1+e^{-z})$. Fungsi kerugian per contoh ialah

$$
\ell(t,z)=-t\log\sigma(z)-(1-t)\log\big[1-\sigma(z)\big].
$$

Turunan terhadap logit memiliki bentuk sederhana:

$$
\frac{\partial\ell}{\partial z}=\sigma(z)-t.
$$

Bentuk turunan ini menjelaskan arah pembaruan gradien: ketika contoh cacat ($t=1$) diberi probabilitas rendah, gradien logit bernilai negatif, sehingga optimisasi cenderung menaikkan logit. `BCEWithLogitsLoss` secara numerik lebih stabil daripada menerapkan sigmoid dan log terpisah.

Regularisasi *weight decay* menambahkan penalti berbasis norma bobot, dan dropout membuat subnetwork efektif berubah selama pelatihan. Kedua teknik tidak menjamin generalisasi ketika perangkat atau distribusi akuisisi berubah.

In [ ]:
# Konfirmasi turunan cross-entropy terhadap logit memakai autograd.
z=torch.tensor([-1.5,0.,1.5],dtype=torch.float64,requires_grad=True)
t=torch.tensor([1.,0.,1.],dtype=torch.float64)
loss=F.binary_cross_entropy_with_logits(z,t,reduction='sum')
loss.backward()
print(pd.DataFrame({'z':z.detach().numpy(),
                    'grad_autograd':z.grad.numpy(),
                    'sigmoid_minus_target':(torch.sigmoid(z)-t).detach().numpy()}).round(5))
assert torch.allclose(z.grad,torch.sigmoid(z)-t,atol=1e-12)

## 7. Arsitektur CNN dan pelatihan

Dua blok konvolusi menghasilkan representasi dari tepi lokal menuju pola cacat yang lebih kompleks. `AdaptiveAvgPool2d` mengubah peta fitur menjadi ringkasan per kanal, sehingga classifier akhir tidak memerlukan parameter sebesar dense layer pada seluruh piksel.

Kita membandingkan tiga konfigurasi yang berbagi dataset, fungsi kerugian, batch size, dan optimizer: (i) CNN tanpa augmentasi, (ii) CNN dengan augmentasi pencahayaan dan geometri, serta (iii) CNN lebih kecil dengan augmentasi sebagai *ablation* kapasitas. Ini **perbandingan terkendali tetapi bukan** eksperimen yang menyamakan jumlah FLOPs atau durasi pelatihan.

In [ ]:
class TabletCNN(nn.Module):
    def __init__(self, width=12, dropout=.15):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,width,3,padding=1),nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width,2*width,3,padding=1),nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(2*width,2*width,3,padding=1),nn.ReLU(),
            nn.AdaptiveAvgPool2d(5))
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(dropout),nn.Linear(2*width*5*5,1))
    def forward(self,x):return self.head(self.features(x)).flatten()

def jumlah_parameter(net):return sum(p.numel() for p in net.parameters() if p.requires_grad)

for w in [6,12]:
    m=TabletCNN(width=w)
    print(f'CNN width={w}: {jumlah_parameter(m):,} parameter; keluaran {tuple(m(torch.zeros(2,1,40,40)).shape)}')

### 7.1 Augmentasi sebagai hipotesis invariansi

Augmentasi tidak sekadar memperbanyak data. Ia menyatakan **hipotesis fisik**: sebuah cacat semestinya tetap dianggap cacat setelah perubahan kecil orientasi, translasi, atau gain. Transformasi yang terlalu kuat bisa merusak label, misalnya memotong cacat kecil hingga tidak terlihat atau membuat citra menyerupai proses akuisisi yang tidak mungkin terjadi.

Kita hanya mengaugmentasi data train. Validation dan test dipertahankan apa adanya untuk evaluasi distribusi normal. Pengujian pergeseran domain menggunakan transformasi tambahan yang ditentukan sebelum membaca metrik akhir.

In [ ]:
def as_tensor(x):return torch.from_numpy(x.astype(np.float32))[:,None]
Xt=as_tensor(images)

def make_aug_batch(x,seed_gen):
    # Transformasi per-citra: rotasi ±9°, translasi ±1.5 piksel, gain dan offset.
    b=x.shape[0]
    angles=seed_gen.uniform(-9,9,b)*np.pi/180
    dx=seed_gen.uniform(-1.5,1.5,b)/(x.shape[-1]/2)
    dy=seed_gen.uniform(-1.5,1.5,b)/(x.shape[-2]/2)
    theta=torch.zeros(b,2,3,dtype=x.dtype,device=x.device)
    theta[:,0,0]=torch.tensor(np.cos(angles),dtype=x.dtype,device=x.device)
    theta[:,0,1]=-torch.tensor(np.sin(angles),dtype=x.dtype,device=x.device)
    theta[:,1,0]=torch.tensor(np.sin(angles),dtype=x.dtype,device=x.device)
    theta[:,1,1]=theta[:,0,0]
    theta[:,0,2]=torch.tensor(dx,dtype=x.dtype,device=x.device)
    theta[:,1,2]=torch.tensor(dy,dtype=x.dtype,device=x.device)
    grid=F.affine_grid(theta,x.size(),align_corners=False)
    transformed=F.grid_sample(x,grid,padding_mode='border',align_corners=False)
    gain=torch.tensor(seed_gen.uniform(.92,1.08,(b,1,1,1)),dtype=x.dtype,device=x.device)
    bias=torch.tensor(seed_gen.uniform(-.025,.025,(b,1,1,1)),dtype=x.dtype,device=x.device)
    return (transformed*gain+bias).clamp(0,1)

fig,ax=plt.subplots(1,4,figsize=(10,3))
ix=int(np.where(labels==1)[0][0]);x=Xt[ix:ix+1].to(DEVICE)
a=[x[0,0].cpu().numpy()]+[make_aug_batch(x,np.random.default_rng(SEED+j))[0,0].cpu().numpy() for j in (11,12,13)]
for i,aa in enumerate(ax):aa.imshow(a[i],cmap='gray',vmin=0,vmax=1);aa.set_title(['Asli','Aug 1','Aug 2','Aug 3'][i]);aa.axis('off')
plt.suptitle('Augmentasi hanya pada train; label harus tetap masuk akal');plt.tight_layout();plt.show()

In [ ]:
def pred_prob_torch(model,index,imgs_tensor=Xt):
    model.eval()
    with torch.inference_mode():
        logits=model(imgs_tensor[index].to(DEVICE))
        return torch.sigmoid(logits).cpu().numpy()

def train_cnn(width=12, augment=False, dropout=.15,seed=SEED,epochs=110,lr=.003):
    torch.manual_seed(seed)
    net=TabletCNN(width=width,dropout=dropout).to(DEVICE)
    optimizer=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=.0005)
    criterion=nn.BCEWithLogitsLoss()
    xtrain=Xt[train_idx].to(DEVICE)
    ytrain=torch.tensor(labels[train_idx],dtype=torch.float32,device=DEVICE)
    rng=np.random.default_rng(seed+7)
    history=[];start=time.perf_counter()
    for epoch in range(epochs):
        net.train()
        permutation=rng.permutation(len(train_idx))
        losses=[]
        for st in range(0,len(permutation),32):
            ii=permutation[st:st+32]
            xb=xtrain[ii];yb=ytrain[ii]
            if augment: xb=make_aug_batch(xb,rng)
            optimizer.zero_grad(set_to_none=True)
            logits=net(xb)
            loss=criterion(logits,yb)
            loss.backward();optimizer.step()
            losses.append((float(loss.item()),len(ii)))
        pval=pred_prob_torch(net,val_idx)
        yhat=(pval>=.5).astype(int)
        history.append({'epoch':epoch+1,'loss':sum(v*n for v,n in losses)/len(train_idx),
                        'val_f1':f1_score(labels[val_idx],yhat,average='macro',zero_division=0),
                        'val_brier':brier_score_loss(labels[val_idx],pval)})
    return net,pd.DataFrame(history),time.perf_counter()-start

# Satu kali lihat bahwa training mengurangi objective pada set kecil, bukan jaminan optimum global.
print('Siap training. GPU:', DEVICE.type=='cuda')

### 7.2 Protokol pemilihan model

Agar hasil tetap dapat dieksekusi pada CPU, notebook menggunakan jumlah epoch moderat dan *fixed training budget*. Ini cocok untuk praktikum metodologi, **bukan** klaim konvergensi optimal semua arsitektur. Model CNN dipilih berdasarkan macro-F1 validasi pada akhir epoch yang telah ditentukan, bukan berdasarkan skor test. Saat nilai validasi sama, dipakai Brier score validasi untuk pemecah seri; pemecah seri ini harus ditetapkan sebelum melihat test.

Untuk studi eksperimental yang lebih kuat, ulangi beberapa seed, laporkan variasi, dan pertimbangkan pembagian data eksternal. Panduan dosen memuat prosedur perluasannya.

In [ ]:
candidates={}
configs=[('CNN dasar',12,False,.15),('CNN augmentasi',12,True,.15),('CNN kecil + augmentasi',6,True,.15)]
for name,width,aug,drop in configs:
    net,hist,seconds=train_cnn(width=width,augment=aug,dropout=drop,seed=SEED,epochs=110)
    candidates[name]={'net':net,'history':hist,'time':seconds,'width':width,'augment':aug,
                      'params':jumlah_parameter(net),
                      'val_f1':float(hist.val_f1.iloc[-1]),'val_brier':float(hist.val_brier.iloc[-1])}
    print(f'{name:24s}: val macro-F1={hist.val_f1.iloc[-1]:.3f} | Brier={hist.val_brier.iloc[-1]:.3f} | {seconds:.1f}s')

### 7.3 Mengapa CNN lambat atau gagal mempelajari cacat kecil?

Pada data terbatas, gradien dari cacat yang hanya meliputi beberapa piksel mudah tenggelam oleh variasi reflektansi latar, noise dan blur. Penggunaan global pooling yang terlalu dini bisa menghilangkan informasi lokasi/ukuran cacat. Karena itu, kita menggunakan pooling adaptif 5×5 yang mempertahankan sebagian struktur spasial, dan memantau kurva validasi selama jumlah epoch tetap yang disepakati.

Penting untuk membedakan **kegagalan optimisasi** (fungsi kerugian train tidak menurun) dan **kegagalan generalisasi** (fungsi kerugian train turun, tetapi validasi buruk). Agar perubahan arsitektur memiliki nilai ilmiah, setiap revisi harus disertai hipotesis, anggaran komputasi, dan data pemilihan yang tepat.

In [ ]:
# Dampak pilihan global pooling terhadap dimensi representasi dan kapasitas.
for width in [6,12]:
    model_tmp=TabletCNN(width)
    ytmp=model_tmp.features(torch.zeros(1,1,40,40))
    print(f'width={width:2d} | feature map={tuple(ytmp.shape)} | total parameter={jumlah_parameter(model_tmp):,}')
assert TabletCNN(12).features(torch.zeros(1,1,40,40)).shape==(1,24,5,5)

## 8. Training dynamics dan model selection

*Training loss* mengukur kecocokan model pada sampel pelatihan. Metrik validasi merupakan diagnostik generalisasi internal, tetapi statistiknya bisa berfluktuasi akibat ukuran data terbatas. Penurunan *training loss* tanpa peningkatan validasi tidak membuktikan model “belajar lebih baik”; dapat menandakan overfitting atau ketidakcocokan metrik.

Skor validasi tetap harus dilaporkan terpisah dari hasil test. Tidak boleh memilih satu konfigurasi dari beberapa run hanya karena memberi test terbaik.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.9))
for name,c in candidates.items():
    h=c['history']; ax[0].plot(h.epoch,h.loss,label=name)
    ax[1].plot(h.epoch,h.val_f1,label=name)
ax[0].set(xlabel='Epoch',ylabel='BCE train',title='Dinamika optimisasi')
ax[1].set(xlabel='Epoch',ylabel='F1 validasi',title='Kinerja validasi',ylim=(-.02,1.04))
for a in ax:a.legend(fontsize=8)
plt.tight_layout();plt.show()

summary_val=[]
for name,c in candidates.items():
    summary_val.append({'model':name,'val_f1':c['val_f1'],'val_brier':c['val_brier'],
                        'parameter':c['params'],'detik_latih':c['time']})
summary_val.append({'model':'Fitur + logistik','val_f1':score_classic,
                    'val_brier':brier_score_loss(labels[val_idx],model_classic.predict_proba(Xtab[val_idx])[:,1]),
                    'parameter':len(model_classic[-1].coef_.ravel())+1,'detik_latih':np.nan})
val_table=pd.DataFrame(summary_val).sort_values(['val_f1','val_brier'],ascending=[False,True])
print(val_table.round(4).to_string(index=False))
selected_name=str(val_table.iloc[0]['model'])
print('\nMODEL TERPILIH berdasarkan validation:',selected_name)

### 8.1 Kejujuran tentang fairness eksperimen

Model klasik menerima **fitur yang dirancang manusia**; CNN menerima piksel, mempelajari fitur dan classifier bersama. Karena tugas keduanya identik, kita dapat membandingkan risiko prediksi pada split yang sama, tetapi **tidak** menyimpulkan arsitektur secara universal lebih unggul: jumlah parameter, jumlah operasi, biaya pengembangan fitur, dan cara tuning berbeda. Perbandingan CPU/GPU juga perlu perangkat yang sama.

*Abalasi* di sini menanyakan dua hal terbatas: apakah augmentasi membantu pada protokol tertentu? Apakah lebar jaringan mengubah hasil dan biaya? Hasil pada satu seed bukan bukti efek kausal yang stabil; ulangi eksperimen di berbagai seed untuk riset formal.

### 8.2 Mengintip representasi CNN: feature maps, bukan alasan kausal

Peta aktivasi menunjukkan area yang menghasilkan respons tinggi pada filter terlatih, tetapi tidak otomatis membuktikan bahwa fitur tersebut kausal atau bahwa model memahami cacat menurut definisi manusia. Kita tampilkan beberapa kanal aktivasi dari CNN augmentasi **tanpa menggunakan label test untuk memilih filter atau mengubah bobot**.

In [ ]:
cnn_demo=candidates['CNN augmentasi']['net'].eval()
ix=int(train_idx[np.flatnonzero(labels[train_idx]==1)[0]])
with torch.inference_mode():
    activ=cnn_demo.features(Xt[ix:ix+1].to(DEVICE)).cpu().numpy()[0]
fig,ax=plt.subplots(1,5,figsize=(12,3))
ax[0].imshow(images[ix],cmap='gray',vmin=0,vmax=1);ax[0].set_title('Citra masukan')
for j in range(4):
    ax[j+1].imshow(activ[j],cmap='magma');ax[j+1].set_title(f'Kanal {j+1}')
for a in ax:a.axis('off')
plt.suptitle('Aktivasi CNN terlatih — contoh TRAIN, bukan bukti interpretabilitas');plt.tight_layout();plt.show()

## 9. Evaluasi akhir yang tidak mencampur validation dan test

Setelah konfigurasi ditentukan, kita melaporkan **semua kandidat** untuk transparansi, tetapi tidak mengubah pilihan berdasarkan hasil test. Akurasi dan F1 bukan pengganti sensitivitas terhadap cacat. Dengan cacat sebagai kelas positif:

$$
\mathrm{Recall}=\frac{TP}{TP+FN},\quad
\mathrm{Precision}=\frac{TP}{TP+FP},\quad
F_1=\frac{2\,\mathrm{Precision}\,\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}.
$$

Confusion matrix akan ditulis dengan baris **label benar**, kolom **prediksi**, dalam urutan `[utuh, cacat]`. Pada inspeksi produk, *false negative* (cacat diprediksi utuh) dapat memiliki biaya yang lebih besar daripada *false positive*.

In [ ]:
def pred_probs(name, idx):
    if name=='Fitur + logistik':return model_classic.predict_proba(Xtab[idx])[:,1]
    return pred_prob_torch(candidates[name]['net'],idx)

def metrics_binary(y,p,threshold=.5):
    pred=(p>=threshold).astype(int)
    return {'accuracy':accuracy_score(y,pred),'balanced_acc':balanced_accuracy_score(y,pred),
            'precision_cacat':precision_score(y,pred,zero_division=0),
            'recall_cacat':recall_score(y,pred,zero_division=0),
            'f1_cacat':f1_score(y,pred,zero_division=0),
            'macro_f1':f1_score(y,pred,average='macro',zero_division=0),
            'brier':brier_score_loss(y,p)}

test_rows=[]; probs_test={}
for name in list(candidates)+['Fitur + logistik']:
    p=pred_probs(name,test_idx);probs_test[name]=p
    test_rows.append({'model':name,**metrics_binary(labels[test_idx],p)})
test_results=pd.DataFrame(test_rows)
print(test_results.round(3).to_string(index=False))
print('\nModel yang dipilih SEBELUM melihat test:',selected_name)

In [ ]:
psel=probs_test[selected_name]
ytrue=labels[test_idx];yhat=(psel>=.5).astype(int)
fig,ax=plt.subplots(1,2,figsize=(10.5,4.1))
ConfusionMatrixDisplay.from_predictions(ytrue,yhat,display_labels=['Utuh','Cacat'],
                                         cmap='Blues',values_format='d',ax=ax[0],colorbar=False)
ax[0].set_title('Test — model yang dipilih dari validation')
wrong=np.flatnonzero(ytrue!=yhat)
if len(wrong):
    i=wrong[np.argmax(np.abs(psel[wrong]-ytrue[wrong]))]
    ax[1].imshow(images[test_idx[i]],cmap='gray',vmin=0,vmax=1)
    ax[1].set_title(f'Kasus salah: benar={ytrue[i]}, p(cacat)={psel[i]:.2f}')
else:
    i=np.argmin(np.abs(psel-.5));ax[1].imshow(images[test_idx[i]],cmap='gray',vmin=0,vmax=1)
    ax[1].set_title(f'Tidak ada kesalahan; contoh borderline p={psel[i]:.2f}')
ax[1].axis('off');plt.tight_layout();plt.show()

### 9.1 Variasi hasil antarbatch pada data test

Akurasi agregat dapat menyembunyikan satu batch yang bermasalah. Oleh karena itu, metrik juga dihitung pada unit batch: setiap batch memuat 12 citra dengan kedua kelas. Hasil ini membantu menemukan pola heterogenitas, tetapi enam batch test masih terlalu sedikit untuk menyimpulkan distribusi populasi.

In [ ]:
by_batch=[]
for b in np.unique(groups[test_idx]):
    sel=groups[test_idx]==b
    by_batch.append({'batch':int(b),'macro_f1':f1_score(ytrue[sel],yhat[sel],average='macro',zero_division=0),
                     'FN':int(np.sum((ytrue[sel]==1)&(yhat[sel]==0))),
                     'FP':int(np.sum((ytrue[sel]==0)&(yhat[sel]==1)))})
by_batch=pd.DataFrame(by_batch)
print(by_batch.to_string(index=False))
ax=by_batch.plot.bar(x='batch',y='macro_f1',legend=False,figsize=(8.5,3.1),ylim=(0,1.03))
ax.set(xlabel='Batch held-out',ylabel='Macro-F1',title='Heterogenitas kinerja model terpilih')
plt.tight_layout();plt.show()

### 9.1 Interpretasi kesalahan

Dua bentuk kesalahan harus dibedakan: **kesalahan klasifikasi**, ketika label prediksi tidak sama dengan label generator, dan **kesalahan semantik**, ketika aturan generator tidak menangkap definisi cacat yang sebenarnya penting bagi manusia. Pada data sintetis, evaluasi pertama dapat dilakukan; evaluasi kedua memerlukan pakar, protokol anotasi, dan citra nyata.

Citra salah klasifikasi dipilih otomatis dari test yang telah dibekukan. Menampilkan contoh kesalahan tidak berarti menggunakannya untuk mengubah model setelah test selesai.

## 10. Probabilitas, kalibrasi, dan aturan keputusan

*Sigmoid output* bukan jaminan probabilitas kejadian yang terkalibrasi. Kita bandingkan proporsi cacat teramati per bin probabilitas dan *Brier score*:

$$
\mathrm{Brier}=\frac1N\sum_{i=1}^N(p_i-t_i)^2.
$$

Untuk probabilitas $p$ dan biaya $C_{FN}$ ketika cacat diloloskan versus $C_{FP}$ ketika tablet utuh ditolak, keputusan optimal **hanya bila probabilitas terkalibrasi dan biaya bersifat tetap** dapat menggunakan ambang

$$
\tau^*=\frac{C_{FP}}{C_{FP}+C_{FN}}.
$$

Ambang tersebut berasal dari membandingkan risiko mengambil keputusan “cacat” ($C_{FP}(1-p)$) dengan keputusan “utuh” ($C_{FN}p$). Jika $C_{FN}>C_{FP}$ maka ambang lebih kecil daripada 0,5. Keputusan aktual bergantung pada biaya dan kebijakan kualitas yang harus ditetapkan bersama pemangku kepentingan, bukan hanya kode.

In [ ]:
def reliability(y,p,bins=np.linspace(0,1,7)):
    inds=np.clip(np.digitize(p,bins)-1,0,len(bins)-2)
    rows=[]
    for b in range(len(bins)-1):
        ix=inds==b
        if ix.any():rows.append((float(p[ix].mean()),float(y[ix].mean()),int(ix.sum())))
    return pd.DataFrame(rows,columns=['prediksi_rata2','frekuensi_cacat','n'])
fig,ax=plt.subplots(1,2,figsize=(10.7,4))
r=reliability(ytrue,psel)
ax[0].plot([0,1],[0,1],'k--',label='Kalibrasi ideal')
ax[0].scatter(r.prediksi_rata2,r.frekuensi_cacat,s=15+8*r.n,alpha=.8,label='Model terpilih')
ax[0].set(xlabel='Rata-rata probabilitas prediksi',ylabel='Fraksi cacat empiris',title='Reliability diagram',xlim=(0,1),ylim=(0,1))
ax[0].legend()
for ratio in [1,3,8]:
    tau=1/(1+ratio)
    cm=confusion_matrix(ytrue,(psel>=tau).astype(int),labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    ax[1].scatter(fp,fn,s=65,label=f'C_FN/C_FP={ratio}, ambang={tau:.2f}')
ax[1].set(xlabel='False positive (test)',ylabel='False negative (test)',title='Trade-off kebijakan biaya')
ax[1].legend(fontsize=8);plt.tight_layout();plt.show()
print('Brier score test:',round(brier_score_loss(ytrue,psel),4))

**Catatan epistemik:** jika data uji terbatas, diagram kalibrasi dapat berfluktuasi dan tidak membenarkan kalibrasi menyeluruh. Kalibrasi formal dapat memerlukan set tersendiri atau validasi silang yang tepat, tanpa menggunakan test akhir untuk *fit* kalibrator.

## 11. Domain shift dan ketahanan model

Pada dunia nyata, perubahan iluminasi, alat, fokus, resolusi, batch bahan, dan protokol akuisisi dapat mengubah distribusi masukan. Kita membuat transformasi *stress* dengan gain, gradien iluminasi, blur, dan noise lebih kuat terhadap **citra test yang sama**. Peningkatan kualitas model pada data normal tidak menjamin ketahanan pada *stress*.

*Stress transform* ditetapkan sebelum mengevaluasi metrik; test tidak boleh dipakai berulang kali untuk menyesuaikan tipe gangguan atau parameter hiper.

In [ ]:
def stres_akuisisi(imgs,seed=SEED+99):
    rng=np.random.default_rng(seed)
    n,h,w=imgs.shape
    yy,xx=np.mgrid[:h,:w]
    result=[]
    for im in imgs:
        gain=rng.uniform(.68,.86)
        offset=rng.uniform(-.085,-.015)
        illum=1+.27*(xx-w/2)/w-.16*(yy-h/2)/h
        pert=ndi.gaussian_filter(im*illum,sigma=1.6,mode='reflect')*gain+offset
        pert+=rng.normal(0,.065,(h,w))
        result.append(np.clip(pert,0,1).astype(np.float32))
    return np.stack(result)

stress_images=stres_akuisisi(images[test_idx])
fig,ax=plt.subplots(2,5,figsize=(11,4.7))
for j in range(5):
    for k,p in enumerate([images[test_idx[j]],stress_images[j]]):
        ax[k,j].imshow(p,cmap='gray',vmin=0,vmax=1)
        ax[k,j].axis('off')
        if j==0: ax[k,j].set_ylabel(['Normal','Shift'][k])
plt.suptitle('Pasangan test normal dan kondisi akuisisi terganggu');plt.tight_layout();plt.show()

In [ ]:
# Model tidak dilatih lagi; Xstress dibuat tanpa label baru.
Xstress=as_tensor(stress_images)
rows_shift=[]
for name in list(candidates)+['Fitur + logistik']:
    if name=='Fitur + logistik':
        p=model_classic.predict_proba(np.stack([fitur_geometri(im) for im in stress_images]))[:,1]
    else:
        model=candidates[name]['net'];model.eval()
        with torch.inference_mode():p=torch.sigmoid(model(Xstress.to(DEVICE))).cpu().numpy()
    m=metrics_binary(ytrue,p)
    rows_shift.append({'model':name,'F1 normal':float(test_results.set_index('model').loc[name,'macro_f1']),
                       'F1 shift':m['macro_f1'],'recall cacat shift':m['recall_cacat'],
                       'Brier shift':m['brier']})
shift_table=pd.DataFrame(rows_shift)
print(shift_table.round(3).to_string(index=False))
ax=shift_table.set_index('model')[['F1 normal','F1 shift']].plot.bar(figsize=(9,4),ylim=(0,1.05))
ax.set(ylabel='Macro-F1',xlabel='',title='Uji ketahanan kondisi akuisisi (tanpa pelatihan ulang)')
plt.xticks(rotation=15,ha='right');plt.tight_layout();plt.show()

## 12. Ketidakpastian: bootstrap tingkat batch

Bootstrap biasa yang mengambil citra secara independen mengabaikan korelasi dalam batch. Kita melakukan **cluster bootstrap**: pilih batch test dengan pengembalian, lalu sertakan semua citra dari batch terpilih. Untuk metrik $T$, distribusi $T^{*(1)},\dots,T^{*(B)}$ merepresentasikan variasi resampling bersyarat pada batch yang tersedia.

$$
\mathrm{CI}_{95\%}^{\rm percentile}=\left[Q_{0.025}(T^*),Q_{0.975}(T^*)\right].
$$

Interval ini hanya bersifat ilustratif karena jumlah batch test kecil, desain simulator terbatas, dan tidak ada sampel perangkat nyata. Ia **tidak** otomatis memperhitungkan ketidakpastian pemilihan model atau variasi inisialisasi CNN.

In [ ]:
unique_test_batches=np.unique(groups[test_idx]); test_group_ids=groups[test_idx]
rng=np.random.default_rng(SEED+119)
bootstrap=[]
for _ in range(500):
    batches=rng.choice(unique_test_batches,size=len(unique_test_batches),replace=True)
    ids=np.concatenate([np.flatnonzero(test_group_ids==b) for b in batches])
    bootstrap.append(f1_score(ytrue[ids],(psel[ids]>=.5).astype(int),average='macro',zero_division=0))
ci=np.quantile(bootstrap,[.025,.975]);point=f1_score(ytrue,yhat,average='macro')
print(f'Macro-F1 test terpilih={point:.3f} | CI cluster bootstrap 95%=[{ci[0]:.3f}, {ci[1]:.3f}]')
fig,ax=plt.subplots(figsize=(8.5,3.6))
ax.hist(bootstrap,bins=18,alpha=.8)
ax.axvline(point,color='black',ls='--',label='Titik estimasi')
for x in ci:ax.axvline(x,color='tab:orange',ls=':',label='Batas CI' if x==ci[0] else None)
ax.set(xlabel='Macro-F1 (resampling batch)',ylabel='Frekuensi',title='Ilustrasi variabilitas metrik tingkat batch')
ax.legend();plt.tight_layout();plt.show()

## 13. Biaya komputasi dan ablation yang dapat ditafsirkan

Dua sumber biaya utama adalah *training* dan *inference*. Jumlah parameter adalah indikator kapasitas yang sederhana, tetapi bukan estimasi langsung FLOPs, memori aktivasi, energi, ataupun latensi di perangkat produksi. Pada CPU, waktu dapat dipengaruhi cache, *threading*, implementasi backend, dan beban mesin.

Ablasi dalam notebook ini memiliki tiga intervensi pada CNN yang sama familinya: (1) tanpa augmentasi, (2) dengan augmentasi, dan (3) lebar kanal lebih kecil dengan augmentasi. Semua menggunakan split yang sama; **efek yang dilihat masih tercampur dengan noise pelatihan satu seed** dan tidak boleh diberi interpretasi signifikan tanpa pengulangan.

In [ ]:
# Inferensi satu batch besar: estimasi indikatif, bukan benchmark hardware formal.
summary_cost=[]
for name,c in candidates.items():
    net=c['net'];net.eval()
    t0=time.perf_counter()
    with torch.inference_mode():
        for _ in range(12): _=net(Xt[test_idx].to(DEVICE))
    ms=(time.perf_counter()-t0)*1000/(12*len(test_idx))
    summary_cost.append({'model':name,'parameter':c['params'],'latensi_ms/citra':ms,
                         'train_s':c['time'],'val_F1':c['val_f1']})
print(pd.DataFrame(summary_cost).round(4).to_string(index=False))
print('Perbandingan inferensi dipengaruhi perangkat dan batch size.')

## 14. Audit reproduksibilitas dan unit test numerik

Beberapa klaim dapat diuji secara otomatis: ukuran output konvolusi, arah gradien *cross-entropy*, isolasi batch train/validation/test, tidak digunakannya metadata label atau batch sebagai fitur, dan keterbatasan input ke rentang numerik yang diharapkan.

Reproduksibilitas yang lebih luas memerlukan catatan versi, prosedur sampling, status determinisme, *source code commit*, protokol evaluasi, dan pernyataan klaim. Bahkan dengan seed tetap, PyTorch menyatakan bahwa keluaran identik tidak selalu dijamin lintas perangkat dan versi: lihat https://docs.pytorch.org/docs/stable/notes/randomness.html.

In [ ]:
# Tes yang otomatis bisa diaudit dari hasil eksekusi.
assert images.shape[0]==len(labels)==len(groups)
assert np.all(np.isfinite(images)) and 0<=images.min()<=images.max()<=1
assert len(set(train_idx)&set(val_idx))==0 and len(set(train_idx)&set(test_idx))==0
assert len(set(groups[train_idx])&set(groups[test_idx]))==0
assert len(set(groups[train_idx])&set(groups[val_idx]))==0
assert set(np.unique(labels[train_idx]))=={0,1}
assert torch.isfinite(next(iter(candidates.values()))['net'](Xt[:2].to(DEVICE))).all()
assert np.isfinite(test_results.select_dtypes(include='number').values).all()
assert ci[0]<=ci[1] and 0<=ci[0]<=ci[1]<=1
print('8 pemeriksaan konsistensi internal: LULUS')

## 15. Tabel simpulan eksperimen

Tuliskan interpretasi berikut berdasar **angka yang baru dihasilkan oleh kernel**, bukan menggunakan klaim yang telah ditetapkan sebelum eksperimen:

- Apakah CNN melampaui baseline fitur pada **validation**? Berapa perbedaan skor dan berapa parameter yang dipakai?
- Model mana yang terpilih dari validation dan bagaimana kinerjanya pada test normal?
- Seberapa besar penurunan macro-F1 saat terjadi *domain shift*? Apakah penurunan sama untuk semua model?
- Apakah perbedaan *training dynamics* konsisten dengan cerita *underfitting* atau *overfitting*? Dapatkah kesimpulan itu ditegakkan dari satu run saja?
- Apakah confidence interval tingkat batch sempit? Apa keterbatasan inferensi dari sedikit batch?
- Kebijakan ambang apa yang masuk akal apabila biaya cacat lolos jauh lebih besar daripada inspeksi ulang tablet utuh?

**Kesimpulan yang diperbolehkan:** model, fitur, dan prosedur pada notebook menunjukkan perilaku tertentu di **dataset sintetis ini**. Kesimpulan tentang real-world inspection, prevalensi cacat, keamanan produk, atau kepatuhan industri **tidak dapat** diperoleh dari hasil simulasi ini.

## 16. Latihan dan proyek berbasis bukti

**L1 — Turunan matematika.** Mulai dari sigmoid dan binary cross-entropy, turunkan $\partial\ell/\partial z$ tanpa menggunakan autodiff. Jelaskan tanda gradien untuk prediksi keliru.

**L2 — Korelasi versus konvolusi.** Tunjukkan pada citra berukuran 4×4 dan kernel 2×2 bahwa keluaran `Conv2d` mengikuti cross-correlation. Kapan pembalikan kernel diperlukan?

**L3 — Dimensi dan kompleksitas.** Hitung keluaran conv untuk input 40×40, kernel 3, padding 1, stride 2; hitung parameter ketika kanal masukan 1 dan keluaran 12. Bandingkan dengan dense layer 1600→32.

**L4 — Kegagalan visual.** Identifikasi setidaknya lima contoh yang salah atau borderline; hipotesiskan penyebab dari blur, cacat kecil, atau pencahayaan. Hindari melakukan tuning dengan test.

**L5 — Feature ablation.** Bandingkan logistic regression menggunakan 4 fitur pertama dengan seluruh 8 fitur. Gunakan validation; jelaskan perbedaan prediksi dan stabilitas terhadap iluminasi.

**L6 — Risiko keputusan.** Misalkan $C_{FN}=8C_{FP}$. Turunkan ambang Bayes yang sesuai dan bandingkan FP/FN tanpa mengubah model.

**L7 — Stabilitas inisialisasi.** Jalankan tiga seed berbeda untuk CNN terpilih. Bandingkan rerata, simpangan baku, dan pengaruh inisialisasi. Jangan menyebut tiga run sebagai estimasi generalisasi lintas perangkat.

**L8 — Pendalaman eksperimen.** Rancang ablation yang mengisolasi dropout, augmentasi, dan lebar kanal satu demi satu, dengan anggaran epoch dan perangkat konsisten.

**L9 — Ketidakpastian terklaster.** Bandingkan CI bootstrap per citra dan bootstrap per batch. Jelaskan mengapa masing-masing mengasumsikan unit independensi berbeda.

**L10 — Audit leakage.** Modifikasi pemisahan sehingga satu batch muncul pada train dan test. Dokumentasikan mengapa desain itu tidak dapat dijadikan evaluasi utama, terlepas dari nilai metrik.

**HOTS 1 — Desain eksperimen:** Usulkan protokol validasi menggunakan batch nyata dari tiga fasilitas produksi, mencakup penetapan label, variasi peralatan, pemisahan operator, ketidakpastian, dan klaim minimum yang boleh dibuat.

**HOTS 2 — Penalaran kritis:** Model A memiliki F1 test lebih tinggi tetapi recall cacat lebih rendah daripada B. Formulasikan fungsi kerugian operasional dan pilih model setelah mendiskusikan prevalensi, biaya, kalibrasi, dan ketidakpastian.

## 17. Syarat validasi sebelum dipakai pada citra nyata

Data dan metode yang telah kita buat tidak dapat langsung diadopsi untuk keputusan mutu farmasi. Sebelum dipakai pada konteks nyata, minimal diperlukan: protokol kamera/iluminasi dan kalibrasi; metadata alat/batch/waktu; sampel tablet dari variasi produksi representatif; ground truth berulang dari ahli; penanganan kelas langka; *external hold-out* fasilitas/perangkat; evaluasi sensitivitas untuk cacat kritis; analisis ketidakpastian; pernyataan keselamatan dan regulasi terkait; serta uji ketahanan *out-of-distribution*.

Ketika label cacat diperoleh dari anotasi manusia, evaluasi harus menyebut *inter-annotator agreement* dan batas label. Citra yang dianggap “cacat” oleh simulator bukan standar mutu industri.

## 18. Rangkuman dan koneksi berikutnya

Pada notebook 10, kita menghubungkan ciri geometri yang dapat ditafsirkan dengan fitur CNN yang dipelajari, menjelaskan operasi konvolusi dan optimisasi, lalu menilai apakah kemajuan model bertahan di luar kondisi pelatihan. Hasil test dan *stress test* harus selalu dibaca berdampingan dengan jenis data, pemilihan model, dan biaya salah klasifikasi.

**Koneksi berikutnya:** pengolahan warna dan kolorimetri memerlukan pemodelan sensor serta variasi iluminasi; perancangan *end-to-end* pipeline perlu menggabungkan evaluasi kuantitatif, kalibrasi dan ketidakpastian; inverse problems akan memperdalam konsekuensi blur dan noise; penelitian deep learning yang matang memerlukan eksperimen berulang, ablation yang benar, dan data eksternal sebelum membuat klaim generalisasi.

**Referensi utama:**

1. Goodfellow, Bengio & Courville (2016), *Deep Learning*, MIT Press (bab CNN dan regularisasi). https://www.deeplearningbook.org/
2. Bishop (2006), *Pattern Recognition and Machine Learning*, Springer (probabilistic classification).
3. Paszke et al. (2019), *PyTorch: An Imperative Style, High-Performance Deep Learning Library*, NeurIPS.
4. Dokumentasi PyTorch Conv2d: https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
5. Dokumentasi PyTorch Reproducibility: https://docs.pytorch.org/docs/stable/notes/randomness.html
6. Dokumentasi scikit-learn GroupShuffleSplit: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html

*Batas penggunaan:* perangkat, batch, label, dan cacat di notebook ini seluruhnya dibangkitkan secara sintetis. Tidak ada klaim kalibrasi sensor maupun kelayakan inspeksi nyata.